# io classification experiment

Classify every moment of a GPS day into one io type, using only **lat, lon and time** from the phone, plus two lookups:

- **place labels** from the io database (`exposure.db`, table `place_labels`): places users bookmarked as Home / Office / School / Other Indoor
- **road centrelines** (Lands Department, 2026-10-01): to tell a train from a car

io types: **Home, Office, School, Other Indoor, Outdoor, MTR, Vehicle** (taxi, bus, minibus, shuttle bus, car, motorcycle are all Vehicle).

Distances between pings are in degrees of **|dlat| + |dlon|** (rule of thumb: 1 m ≈ 0.00001 deg); distances to roads and place labels are in metres.

Contents:
1. Basic features (1011): lat, lon, time, delta t, delta distance, velocity
2. Inputs: place labels, road centrelines
3. Thresholds
4. Pipeline
5. Results: 1011
6. Results: 1012
7. Evidence: vehicle vs MTR by distance to road
8. Output: client points and hourly API rows
9. Decision rule (summary)
10. Appendix: data checked and not used

## 1. Basic features (1011)

One row per ping. Row *i* holds the change from ping *i-1* (p1) to ping *i* (p2), so the first row has no delta (`NaN`):

- `delta_t_s` = t2 - t1, seconds
- `delta_dist_deg` = |lat2 - lat1| + |lon2 - lon1|, degrees
- `v_deg_s` = delta_dist_deg / delta_t_s (`NaN` when delta_t_s = 0)

In [ ]:
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from pyproj import Transformer

# This notebook lives in testing/, one level below the repo root.
REPO_ROOT = Path.cwd().parent
SAMPLES = REPO_ROOT / "data" / "samples"
GPX_NS = {"gpx": "http://www.topografix.com/GPX/1/1"}

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)


def read_gpx(path):
    # lat, lon, time (HK local) of every <trkpt>; all other GPX fields are ignored.
    root = ET.parse(path).getroot()
    rows = [
        {"lat": float(pt.get("lat")), "lon": float(pt.get("lon")), "time": pt.findtext("gpx:time", namespaces=GPX_NS)}
        for pt in root.iterfind(".//gpx:trkpt", GPX_NS)
    ]
    d = pd.DataFrame(rows)
    d["time"] = pd.to_datetime(d["time"], utc=True).dt.tz_convert("Asia/Hong_Kong")
    return d


def add_deltas(d):
    d["delta_t_s"] = d["time"].diff().dt.total_seconds()
    d["delta_dist_deg"] = d["lat"].diff().abs() + d["lon"].diff().abs()
    d["v_deg_s"] = d["delta_dist_deg"] / d["delta_t_s"].replace(0, np.nan)
    return d

In [ ]:
df = add_deltas(read_gpx(SAMPLES / "1011_route_history.gpx"))
print(f"{len(df)} pings, {df['time'].min()} -> {df['time'].max()}, sorted: {df['time'].is_monotonic_increasing}")
print(f"pings with delta_t_s = 0: {(df['delta_t_s'] == 0).sum()}")
df.head(10)

In [ ]:
df[["delta_t_s", "delta_dist_deg", "v_deg_s"]].describe()

## 2. Inputs

### Place labels (io database)
Read-only from the repo-root `exposure.db`.

In [ ]:
with sqlite3.connect(f"file:{(REPO_ROOT / 'exposure.db').as_posix()}?mode=ro", uri=True) as conn:
    LABELS = pd.read_sql("SELECT label_id, category, lat, lng, radius_m FROM place_labels", conn)
LABELS

### Road centrelines
`data/samples/road_network/road_centreline_20261001/RoadCentreLine.geojson`, Lands Department via CSDI
([Road Centreline](https://portal.csdi.gov.hk/csdi-webpage/dataset/landsd_rcd_1637310758814_80061),
`file-api?dataset_id=landsd_rcd_1637310758814_80061&format=geojson&layer_name=RoadCentreLine`).
Data specification: `Data_Specifications_Transportation_EN.pdf` in the same folder.

In [ ]:
rcl = gpd.read_file(SAMPLES / "road_network" / "road_centreline_20261001" / "RoadCentreLine.geojson").to_crs(2326)  # metres
print(f"{len(rcl)} centrelines, last update {rcl['LASTUPDATEDATE'].max()}")
print(rcl["STREETTYPE"].value_counts().to_string())

RCL_TREE = shapely.STRtree(rcl.geometry.values)
TO_HK1980 = Transformer.from_crs(4326, 2326, always_xy=True)

## 3. Thresholds

In [ ]:
# deg = |dlat| + |dlon|; 1 m ~ 0.00001 deg
DEG_PER_M = 0.00001

# Step 0: spikes
SPIKE_MIN_DEG = 0.001           # ~100 m: both legs of a spike are at least this long ...
SPIKE_RETURN_RATIO = 0.25       # ... and the ping after it is back within this share of the shorter leg

# Step 1: motion per interval
NOISE_DEG = 0.0003              # ~30 m: GPS jitter floor; net displacement below this = not moving
WINDOW_S = 120                  # v_net window, centred on each ping
GAP_S = 120                     # an interval at least this long had no pings in between
JUMP_DEG = 0.003                # ~300 m in one interval: travelled while GPS was dark
STILL_MAX_V = 0.000005          # deg/s, ~0.5 m/s
WALK_MAX_V = 0.00003            # deg/s, ~3 m/s (walking is 1-1.5 m/s)

# Step 2: runs clean-up
STOP_MAX_S = 180                # still/walk this short between two vehicle runs = a stop inside the trip
MIN_RUN_S = {"vehicle": 120, "walk": 60}  # shorter runs are GPS blips
SAME_PLACE_DEG = 0.0008         # ~80 m: two still runs this close are the same place ...
SAME_PLACE_MAX_WALK_S = 300     # ... if the walk between them is at most this long

# Step 3: MTR
RAIL_MIN_ROAD_M = 15            # vehicle run with median distance to the nearest road centreline >= this = rail
MTR_MAX_AVG_V = 0.00015         # deg/s, ~15 m/s door to door; a faster jump started earlier
INTERCHANGE_MAX_S = 10 * 60     # still/walk between two MTR runs this short = changing lines in a station

# Step 4: still runs
LABEL_DRIFT_M = 30              # place label circle grown by this for indoor GPS drift
LABEL_MIN_PING_FRACTION = 0.5   # share of the run's pings inside the circle
INDOOR_MIN_S = 20 * 60          # shorter still runs (without a label) are Outdoor
HOME_HOURS = (0, 6)             # [start, end) HK hour: a long still run overlapping this is Home
HOME_RADIUS_DEG = 0.001         # ~100 m: other still runs this close to a night stay are Home too

# Step 5: output points
STAY_MOTIONS = ("still", "interchange")     # one point each (median of the pings): removes indoor / in-station jitter
ROUTE_TOL_M = 15               # thinned route stays within this many metres of the GPS path (Douglas-Peucker)
ROUTE_MAX_GAP_DEG = 0.003       # ~300 m: keep a route point at least this often
ROAD_NAME_MAX_M = 30            # a Vehicle point names its road when a centreline is this close
HK_TIME_FORMAT = "%Y%m%d%H%M%S" # expo_calx t, as in app/config/constants.py
DELTA_T_DECIMALS = 6

## 4. Pipeline

`classify_day(path)` -> `(pings, runs)`:

- **Step 0** remove spikes
- **Step 1** motion per interval: `jump` / `still` / `walk` / `vehicle`, using `v_net` = net displacement over a 120 s window minus the jitter floor
- **Step 2** consecutive intervals with the same motion form a run; clean up stops, blips, same-place stays
- **Step 3** MTR: jumps, rail test (distance to road), time extension, interchanges
- **Step 4** decision tree per run -> io

Why not classify each ping on its own: indoor GPS jitter moves a still phone by ~0.0001-0.0002 deg (10-20 m) per ping whatever the delta t,
so with 1 s between pings a still phone looks like 15 m/s. `v_net` cancels that out.

### Helpers

In [ ]:
def deg(a_lat, a_lon, b_lat, b_lon):
    return abs(a_lat - b_lat) + abs(a_lon - b_lon)


def remove_spikes(d):
    # Drop ping b of a, b, c when a->b and b->c are both long and c is back near a. Re-scan after each drop.
    la, lo = d["lat"].to_numpy(), d["lon"].to_numpy()
    keep = list(range(len(d)))
    i = 1
    while i < len(keep) - 1:
        a, b, c = keep[i - 1], keep[i], keep[i + 1]
        out, back = deg(la[a], lo[a], la[b], lo[b]), deg(la[b], lo[b], la[c], lo[c])
        if min(out, back) >= SPIKE_MIN_DEG and deg(la[a], lo[a], la[c], lo[c]) <= SPIKE_RETURN_RATIO * min(out, back):
            keep.pop(i)
            i = max(i - 1, 1)
        else:
            i += 1
    return d.iloc[keep].reset_index(drop=True)


def add_road_distance(d):
    # road_m = metres to the nearest road centreline; road_type / road_name = its STREETTYPE / English name.
    x, y = TO_HK1980.transform(d["lon"].to_numpy(), d["lat"].to_numpy())
    idx, dist = RCL_TREE.query_nearest(shapely.points(x, y), return_distance=True, all_matches=False)
    d["road_m"] = dist
    d["road_type"] = rcl["STREETTYPE"].to_numpy()[idx[1]]
    d["road_name"] = rcl["ENGLISHSTREETNAME"].to_numpy()[idx[1]]
    return d


def to_runs(labels):
    # Consecutive rows with the same label -> list of [label, first_row, last_row].
    runs = []
    for i, lab in enumerate(labels):
        if runs and runs[-1][0] == lab:
            runs[-1][2] = i
        else:
            runs.append([lab, i, i])
    return runs


def neighbours(runs, n):
    return (runs[n - 1] if n > 0 else None), (runs[n + 1] if n + 1 < len(runs) else None)


def match_label(pts):
    # Best place label holding >= LABEL_MIN_PING_FRACTION of the pings: most pings inside, then smaller radius.
    best = None
    for lab in LABELS.itertuples():
        r_deg = (lab.radius_m + LABEL_DRIFT_M) * DEG_PER_M
        frac = ((pts["lat"] - lab.lat).abs() + (pts["lon"] - lab.lng).abs() <= r_deg).mean()
        if frac >= LABEL_MIN_PING_FRACTION and (best is None or (frac, -lab.radius_m) > (best[0], -best[1].radius_m)):
            best = (frac, lab)
    return best[1].category if best else None


def overlaps_night(start, end):
    for day in pd.date_range(start.normalize(), end.normalize(), freq="D"):
        if start < day + pd.Timedelta(hours=HOME_HOURS[1]) and end > day + pd.Timedelta(hours=HOME_HOURS[0]):
            return True
    return False

### classify_day

In [ ]:
def classify_day(path):
    d = add_road_distance(add_deltas(remove_spikes(read_gpx(path))))
    t = (d["time"] - d["time"].iloc[0]).dt.total_seconds().to_numpy()
    la, lo = d["lat"].to_numpy(), d["lon"].to_numpy()
    secs = d["delta_t_s"].fillna(0).to_numpy()
    road_m = d["road_m"].to_numpy()

    # ---- Step 1: motion per interval ----
    j = np.searchsorted(t, t - WINDOW_S / 2, side="left")
    k = np.searchsorted(t, t + WINDOW_S / 2, side="right") - 1
    span = t[k] - t[j]
    disp = np.abs(la[k] - la[j]) + np.abs(lo[k] - lo[j])
    d["v_net_deg_s"] = np.where(span > 0, np.maximum(disp - NOISE_DEG, 0) / np.where(span > 0, span, 1), np.nan)
    v_int = pd.concat([d["v_net_deg_s"], d["v_net_deg_s"].shift()], axis=1).mean(axis=1)  # mean of the 2 end pings
    d["motion"] = np.select(
        [d["delta_dist_deg"] >= JUMP_DEG, d["delta_t_s"] >= GAP_S, v_int.isna() | (v_int < STILL_MAX_V), v_int < WALK_MAX_V],
        ["jump", "still", "still", "walk"],
        default="vehicle",
    )
    d.loc[0, "motion"] = d.loc[1, "motion"]  # row 0 has no interval

    # ---- Step 2: runs and clean-up ----
    def run_s(r):
        return secs[r[1]:r[2] + 1].sum()

    def centre(r):
        a = max(r[1] - 1, 0)
        return np.median(la[a:r[2] + 1]), np.median(lo[a:r[2] + 1])

    def clean_once(labels):
        runs = to_runs(labels)
        # 2a. a short stop between two fast runs is part of the trip (traffic light, bus stop)
        for n, r in enumerate(runs):
            prev, nxt = neighbours(runs, n)
            if (r[0] in ("still", "walk") and prev and nxt and run_s(r) <= STOP_MAX_S
                    and {prev[0], nxt[0]} <= {"vehicle", "jump"}):
                labels[r[1]:r[2] + 1] = "vehicle" if "vehicle" in (prev[0], nxt[0]) else "jump"
                return True
        # 2b. the shortest GPS blip joins a neighbour (a short vehicle burst prefers a jump next to it)
        blips = [(run_s(r), n) for n, r in enumerate(runs) if r[0] in MIN_RUN_S and run_s(r) < MIN_RUN_S[r[0]]]
        for _, n in sorted(blips):
            r = runs[n]
            cands = [x for x in neighbours(runs, n) if x]
            if not cands:
                continue
            jumps = [x for x in cands if x[0] == "jump"]
            target = jumps[0] if r[0] == "vehicle" and jumps else max(cands, key=run_s)
            labels[r[1]:r[2] + 1] = target[0]
            return True
        return False

    def merge_same_place(labels):
        # 2c. still + short walk + still at the same place -> one stay (walking around inside a mall)
        runs = to_runs(labels)
        for a, w, b in zip(runs, runs[1:], runs[2:]):
            if (a[0] == b[0] == "still" and w[0] == "walk" and run_s(w) <= SAME_PLACE_MAX_WALK_S
                    and deg(*centre(a), *centre(b)) <= SAME_PLACE_DEG):
                labels[w[1]:w[2] + 1] = "still"
                return True
        return False

    labels = d["motion"].to_numpy().copy()
    while clean_once(labels) or merge_same_place(labels):
        pass

    # 2d. a jump next to a vehicle run is GPS loss during the same road trip
    runs = to_runs(labels)
    for n, r in enumerate(runs):
        if r[0] == "jump" and "vehicle" in [x[0] for x in neighbours(runs, n) if x]:
            labels[r[1]:r[2] + 1] = "vehicle"

    # ---- Step 3: MTR ----
    # 3a. rail test: a vehicle run that keeps away from the road centrelines is above-ground MTR
    for r in to_runs(labels):
        if r[0] == "vehicle" and np.median(road_m[max(r[1] - 1, 0):r[2] + 1]) >= RAIL_MIN_ROAD_M:
            labels[r[1]:r[2] + 1] = "rail"

    # 3b. a jump too fast for MTR started earlier (stale pings): take in the still intervals just before it
    for r in to_runs(labels):
        if r[0] != "jump":
            continue
        a, b = max(r[1] - 1, 0), r[2]
        need_s = deg(la[a], lo[a], la[b], lo[b]) / MTR_MAX_AVG_V
        took_s, i = run_s(r), r[1] - 1
        while took_s < need_s and i >= 1 and labels[i] == "still":
            labels[i] = "jump"
            took_s += secs[i]
            i -= 1

    # 3c. interchange: a short still/walk run between two MTR runs is inside a station
    runs = to_runs(labels)
    for n, r in enumerate(runs):
        prev, nxt = neighbours(runs, n)
        if (r[0] in ("still", "walk") and prev and nxt and run_s(r) <= INTERCHANGE_MAX_S
                and prev[0] in ("jump", "rail") and nxt[0] in ("jump", "rail")):
            labels[r[1]:r[2] + 1] = "interchange"

    d["run_motion"] = labels
    d["run_id"] = (d["run_motion"] != d["run_motion"].shift()).cumsum()

    # ---- Step 4: features per run + decision tree ----
    def run_features(g):
        first = max(g.index[0] - 1, 0)  # a run starts at the ping before its first interval
        pts = d.loc[first:g.index[-1]]
        motion = g["run_motion"].iloc[0]
        return pd.Series({
            "motion": motion,
            "start": pts["time"].iloc[0],
            "end": pts["time"].iloc[-1],
            "minutes": g["delta_t_s"].fillna(0).sum() / 60,
            "n_pings": len(g),
            "road_med_m": pts["road_m"].median() if motion in ("vehicle", "rail") else np.nan,
            "lat": pts["lat"].median(),
            "lon": pts["lon"].median(),
            "label": match_label(pts) if motion == "still" else None,
            "night": motion == "still" and overlaps_night(pts["time"].iloc[0], pts["time"].iloc[-1]),
        })

    runs_df = d.groupby("run_id").apply(run_features)
    night_homes = runs_df[runs_df["night"] & (runs_df["minutes"] * 60 >= INDOOR_MIN_S)][["lat", "lon"]].to_numpy()

    def classify(run):
        # Returns (io, reason).
        m = run["motion"]
        if m == "jump":
            return "MTR", "jump"
        if m == "rail":
            return "MTR", "rail (far from roads)"
        if m == "interchange":
            return "MTR", "interchange"
        if m == "vehicle":
            return "Vehicle", "vehicle"
        if m == "walk":
            return "Outdoor", "walk"
        # still
        if pd.notna(run["label"]):
            return run["label"], "place label"
        long_stay = run["minutes"] * 60 >= INDOOR_MIN_S
        if run["night"] and long_stay:
            return "Home", "night stay"
        if any(deg(run["lat"], run["lon"], h_lat, h_lon) <= HOME_RADIUS_DEG for h_lat, h_lon in night_homes):
            return "Home", "same place as a night stay"
        if not long_stay:
            return "Outdoor", "short stop"
        return "Other Indoor", "long stay"

    runs_df[["io", "reason"]] = runs_df.apply(classify, axis=1, result_type="expand")
    d["io"] = d["run_id"].map(runs_df["io"])
    return d, runs_df


def show_runs(runs_df):
    s = runs_df.copy()
    s["start"] = s["start"].dt.strftime("%m-%d %H:%M:%S")
    s["end"] = s["end"].dt.strftime("%m-%d %H:%M:%S")
    return s[["io", "reason", "motion", "start", "end", "minutes", "n_pings", "road_med_m", "lat", "lon"]].round(
        {"minutes": 1, "road_med_m": 1, "lat": 5, "lon": 5})


def minutes_per_io(d):
    return d.groupby("io")["delta_t_s"].sum().div(60).round(1).sort_values(ascending=False).rename("minutes")

## 5. Results: 1011

In [ ]:
pings_1011, runs_1011 = classify_day(SAMPLES / "1011_route_history.gpx")
print(f"{len(pings_1011)} pings after spike removal")
show_runs(runs_1011)

In [ ]:
minutes_per_io(pings_1011)

## 6. Results: 1012

In [ ]:
pings_1012, runs_1012 = classify_day(SAMPLES / "1012_route_history.gpx")
print(f"{len(pings_1012)} pings after spike removal")
show_runs(runs_1012)

In [ ]:
minutes_per_io(pings_1012)

## 7. Evidence: vehicle vs MTR by distance to road

A car / bus drives **on** a road centreline (a few metres off); a train runs on its own track **beside** the roads.
Vehicle and rail runs of both days, with the median and p90 distance of their pings to the nearest road centreline:

In [ ]:
def road_evidence(pings, runs_df):
    moving = runs_df[runs_df["motion"].isin(["vehicle", "rail"])]
    p90 = pings.groupby("run_id")["road_m"].quantile(0.9).rename("road_p90_m").round(1)
    return show_runs(moving)[["io", "start", "end", "minutes", "road_med_m"]].join(p90)


pd.concat({"1011": road_evidence(pings_1011, runs_1011), "1012": road_evidence(pings_1012, runs_1012)}, names=["day", "run_id"])

## 8. Output: client points and hourly API rows

The client should not get every ping, and expo_calx scores each row against that hour's AQHI. So each day becomes two tables:

**Client points** (what the app draws):
- a **stay** (still run: Home, Office, School, Other Indoor, or an Outdoor short stop) and an **MTR interchange** (changing lines inside a station)
  → **1 point** at the median of its pings, which also removes the indoor / in-station GPS jitter
- a **walk / vehicle / MTR run** → its pings thinned by **Douglas-Peucker** (the thinned line stays within 15 m of the GPS path,
  about a road's width, so every turn, ramp and junction is kept) **plus at least one point every ~300 m** on long straight roads;
  **both ends of every GPS gap are always kept**, so an underground MTR ride always keeps its two stations
  (only the redundant 1-second pings where GPS comes back are dropped)
- every route point has `heading_deg` (0 = north, clockwise) and `compass` towards the next point, so the client can draw the direction;
  a Vehicle point also has `road_name` (the centreline within 30 m), e.g. which highway it is on

**Hourly API rows** (what goes to expo_calx): every client point owns the time until the next point; that time is cut at HK clock hours
(as `app/etl/time_split.py` does) and each piece is one row `[t, pid, lng, lat, {"IO": io}, delta_t_hours]`.
The last point of a route before a stay only closes the line, owns no time and gets no API row.

Why Douglas-Peucker and not "one point every 100 m": on these two days (6 vehicle rides, 2351 pings) one point every 100 m kept 390 points
but cut corners by up to 72 m; Douglas-Peucker kept 278 points at 10 m and 175 at 20 m, never more than that far from the GPS path.
15 m is between the two and also drops most back-and-forth points from GPS jitter while a vehicle or train is stopped.
The ~300 m minimum spacing only adds points on long straight stretches.

In [ ]:
COMPASS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]


def dp_keep(x, y, tol):
    # Douglas-Peucker on metres: True for the points to keep so the line stays within tol of every point.
    keep = np.zeros(len(x), bool)
    keep[0] = keep[-1] = True
    stack = [(0, len(x) - 1)]
    while stack:
        a, b = stack.pop()
        if b <= a + 1:
            continue
        px, py = x[a + 1:b], y[a + 1:b]
        dx, dy = x[b] - x[a], y[b] - y[a]
        seg2 = dx * dx + dy * dy
        u = np.clip(((px - x[a]) * dx + (py - y[a]) * dy) / seg2, 0, 1) if seg2 else np.zeros(len(px))
        dist = np.hypot(px - (x[a] + u * dx), py - (y[a] + u * dy))
        i = int(np.argmax(dist))
        if dist[i] > tol:
            m = a + 1 + i
            keep[m] = True
            stack += [(a, m), (m, b)]
    return keep


def every_keep(lat, lon, step_deg):
    # True for a point once it is step_deg (|dlat| + |dlon|) away from the last kept one.
    keep = np.zeros(len(lat), bool)
    keep[0] = keep[-1] = True
    last = 0
    for i in range(1, len(lat)):
        if deg(lat[i], lon[i], lat[last], lon[last]) >= step_deg:
            keep[i] = True
            last = i
    return keep


def headings(x, y):
    # Bearing (deg, 0 = north, clockwise) from each point to the next; the last point repeats the previous one.
    if len(x) < 2:
        return np.full(len(x), np.nan)
    dx, dy = np.diff(x), np.diff(y)
    h = (np.degrees(np.arctan2(dx, dy)) + 360) % 360
    h[(dx == 0) & (dy == 0)] = np.nan
    return np.append(h, h[-1])


def build_points(pings, runs_df):
    # Client points: 1 per stay, thinned pings per route run. Each point owns [start, end).
    x_all, y_all = TO_HK1980.transform(pings["lon"].to_numpy(), pings["lat"].to_numpy())
    la, lo, times = pings["lat"].to_numpy(), pings["lon"].to_numpy(), pings["time"]
    gap_end = (pings["delta_dist_deg"] >= JUMP_DEG).to_numpy()  # ping right after a GPS-dark jump
    run_rows = pings.groupby("run_id").indices
    runs_list = list(runs_df.itertuples())
    out = []
    for n, run in enumerate(runs_list):
        rows = run_rows[run.Index]
        first, last = max(rows[0] - 1, 0), rows[-1]  # a run starts at the ping before its first interval
        base = {"run_id": run.Index, "io": run.io, "motion": run.motion}
        if run.motion in STAY_MOTIONS:
            out.append({**base, "kind": "stay", "start": run.start, "end": run.end,
                        "lat": np.median(la[first:last + 1]), "lon": np.median(lo[first:last + 1])})
            continue
        idx = np.arange(first, last + 1)
        keep = dp_keep(x_all[idx], y_all[idx], ROUTE_TOL_M) | every_keep(la[idx], lo[idx], ROUTE_MAX_GAP_DEG)
        gap = np.flatnonzero(gap_end[idx])  # both ends of every GPS gap (e.g. the 2 MTR stations) are always kept
        keep[gap] = True
        keep[gap[gap > 0] - 1] = True
        idx = idx[keep]
        head = headings(x_all[idx], y_all[idx])
        ends = list(times.iloc[idx[1:]]) + [run.end]
        nxt = runs_list[n + 1] if n + 1 < len(runs_list) else None
        if nxt is not None and nxt.motion not in STAY_MOTIONS:
            idx, head, ends = idx[:-1], head[:-1], ends[:-1]  # the next route run starts with this same ping
        for i, h, end in zip(idx, head, ends):
            road = pings.at[i, "road_name"] if run.io == "Vehicle" and pings.at[i, "road_m"] <= ROAD_NAME_MAX_M else None
            out.append({**base, "kind": "route", "start": times.iloc[i], "end": end, "lat": la[i], "lon": lo[i],
                        "heading_deg": h, "compass": COMPASS[int(((h + 22.5) % 360) // 45)] if pd.notna(h) else None,
                        "road_name": road})
    points = pd.DataFrame(out)
    points["minutes"] = (points["end"] - points["start"]).dt.total_seconds() / 60
    return points[["run_id", "io", "kind", "motion", "start", "end", "minutes", "lat", "lon", "heading_deg", "compass", "road_name"]]


def split_hourly(points):
    # Cut each point's [start, end) at HK clock hours; zero-length pieces are dropped.
    rows = []
    for p in points.itertuples():
        hour = p.start.floor("h")
        while hour < p.end:
            nxt = hour + pd.Timedelta(hours=1)
            s, e = max(p.start, hour), min(p.end, nxt)
            if e > s:
                rows.append({"point": p.Index, "hour_start_hk": hour, "start_time_hk": s, "io": p.io,
                             "lat": p.lat, "lon": p.lon, "delta_t_hours": (e - s).total_seconds() / 3600})
            hour = nxt
    return pd.DataFrame(rows)


def request_rows(hourly, pid):
    # expo_calx rows, same shape as app/etl/request_rows.py: [t, pid, lng, lat, {"IO": io}, delta_t_hours]
    return [[r.start_time_hk.strftime(HK_TIME_FORMAT), pid, r.lon, r.lat, {"IO": r.io}, round(r.delta_t_hours, DELTA_T_DECIMALS)]
            for r in hourly.itertuples()]


def show_points(points):
    s = points.copy()
    s["start"] = s["start"].dt.strftime("%m-%d %H:%M:%S")
    s["end"] = s["end"].dt.strftime("%m-%d %H:%M:%S")
    return s.round({"minutes": 2, "lat": 5, "lon": 5, "heading_deg": 0})


points_1011, points_1012 = build_points(pings_1011, runs_1011), build_points(pings_1012, runs_1012)
hourly_1011, hourly_1012 = split_hourly(points_1011), split_hourly(points_1012)

### How many points / rows, per io

In [ ]:
def counts(pings, points, hourly):
    return pd.DataFrame({"pings": pings.groupby("io").size(), "client_points": points.groupby("io").size(),
                         "api_rows": hourly.groupby("io").size()})


count_table = pd.concat({"1011": counts(pings_1011, points_1011, hourly_1011),
                         "1012": counts(pings_1012, points_1012, hourly_1012)}, axis=1).fillna(0).astype(int)
count_table.loc["total"] = count_table.sum()
count_table

### Check: no time is lost (minutes per io, pings vs API rows)

In [ ]:
pd.concat({
    "1011 pings": minutes_per_io(pings_1011), "1011 api_rows": hourly_1011.groupby("io")["delta_t_hours"].sum().mul(60).round(1),
    "1012 pings": minutes_per_io(pings_1012), "1012 api_rows": hourly_1012.groupby("io")["delta_t_hours"].sum().mul(60).round(1),
}, axis=1)

### Client points, 1011: stays

In [ ]:
show_points(points_1011[points_1011["kind"] == "stay"]).drop(columns=["heading_deg", "compass", "road_name"])

### Client points, 1011: the taxi ride (14:53-15:05) with direction and road

In [ ]:
show_points(points_1011[points_1011["run_id"] == 8])

### Client points, 1012: the East Rail ride University → Tai Wai (21:14-21:23)

In [ ]:
show_points(points_1012[points_1012["run_id"] == 14])

### Hourly totals (minutes per HK hour and io), 1011: the stacked bar chart

In [ ]:
hourly_1011.pivot_table(index=hourly_1011["hour_start_hk"].dt.strftime("%H:00"), columns="io",
                        values="delta_t_hours", aggfunc="sum").mul(60).round(1).fillna(0)

### expo_calx request rows, 1011 (first 10)

In [ ]:
rows_1011 = request_rows(hourly_1011, "1011")
print(f"{len(rows_1011)} rows")
for row in rows_1011[:10]:
    print(row)

## 9. Decision rule (summary)

### Step 0: remove spikes
Drop ping B of A, B, C when A→B and B→C are both >= 0.001 deg (~100 m) and C is back within 25% of the shorter leg from A. Re-scan after each drop.

### Step 1: motion of each interval (ping *i-1* → ping *i*)
`v_net` of a ping = (net |dlat|+|dlon| between the first and last ping within ±60 s − 0.0003 deg jitter floor, min 0) / their time span.
An interval's `v_net` = mean of its two end pings. First match wins:

| motion | condition |
|---|---|
| `jump` | the interval moved >= 0.003 deg (~300 m): travelled while GPS was dark |
| `still` | the interval is >= 120 s long (no pings in between), or `v_net` < 0.000005 deg/s (~0.5 m/s) |
| `walk` | `v_net` < 0.00003 deg/s (~3 m/s) |
| `vehicle` | otherwise |

### Step 2: runs and clean-up
Consecutive intervals with the same motion form a run. Repeat until nothing changes:
- **2a** still/walk run <= 3 min between two vehicle/jump runs → part of the trip (traffic light, bus stop)
- **2b** vehicle run < 2 min or walk run < 1 min → GPS blip, joins the longer neighbour (a vehicle blip next to a jump joins the jump)
- **2c** still + walk <= 5 min + still, with the two still centres <= 0.0008 deg (~80 m) apart → one stay

Then **2d**: a jump next to a vehicle run → vehicle (GPS loss during the same road trip).

### Step 3: MTR
- **3a rail test**: a vehicle run whose pings have a **median distance to the nearest road centreline >= 15 m** → MTR (above ground).
  Road trips so far: 2-6 m; East Rail line: 24 m.
- **3b time extension**: a jump faster than 0.00015 deg/s (~15 m/s door to door) started earlier (stale pings):
  the still intervals right before it join the MTR trip, newest first, until the speed is plausible.
- **3c interchange**: a still/walk run <= 10 min between two MTR runs → MTR (changing lines in a station).

### Step 4: decision tree per run
```
run
├─ jump ............................................. MTR      (underground)
├─ vehicle
│   ├─ median distance to road >= 15 m ............... MTR      (above ground)
│   └─ otherwise ..................................... Vehicle  (taxi, bus, minibus, shuttle bus, car, motorcycle)
├─ still/walk <= 10 min between two MTR runs ......... MTR      (interchange)
├─ walk .............................................. Outdoor
└─ still
    ├─ inside a place label .......................... the label's category (Home / Office / School / Other Indoor)
    │    (>= 50% of its pings within the label radius + 30 m)
    ├─ >= 20 min and overlaps 00:00-05:59 ............ Home     (night stay)
    ├─ within 100 m of a night stay of the same file . Home
    ├─ < 20 min ...................................... Outdoor  (short stop, e.g. waiting)
    └─ otherwise ..................................... Other Indoor
```
- Place labels come first: a labelled place keeps its category even at night (a night at the office is Office).
- Office and School are never guessed: without a label they are Other Indoor.

### Step 5: output (section 8)
- **Client points**: stay or MTR interchange → 1 point (median of its pings); walk / vehicle / MTR → Douglas-Peucker 15 m
  + a point at least every 0.003 deg (~300 m) + both ends of every GPS gap (MTR stations). Route points carry `heading_deg` / `compass`
  (to the next point) and Vehicle points `road_name`.
- **Hourly API rows**: each point owns the time until the next point, cut at HK clock hours → `[t, pid, lng, lat, {"IO": io}, delta_t_hours]`.
  Total time per io is unchanged.

### Known limits
- The 15 m rail threshold comes from 2 East Rail rides; other lines are untested.
- A ferry is far from every road, so it would come out as MTR (ferry is not an io type).
- Indoor stays < 20 min (e.g. a quick shop visit) come out as Outdoor; walking through a station is Outdoor.
- An unlabelled school or office (e.g. a university classroom) is Other Indoor.

### Checked against the user's own account of 1011 and 1012
1011: every run matches. 1012: every run matches except 18:27-21:08 (university classroom, no label → Other Indoor).

## 10. Appendix: data checked and not used

- **Planning Department land use raster (LUHK 2025)**: no class code table is published with it, and the pixel codes at known places did not match
  (a road, a public estate and woodland shared codes); mixed-use buildings make it unreliable anyway.
- **Road network 100 m buffer** (`road_network/RoadCentreline_Buffer_SHP/RN_buffer100m.shp`, from the 2025-02-03 centrelines):
  almost every ping of every io type is inside it, including the East Rail line, so it separates nothing; it also misses the Central Kowloon Route.
  The centrelines above (2026-10-01, distances in metres) replace it.
- **Straightness + speed** as the rail test: worked on the 2 East Rail rides, but a straight highway next to a railway (Tolo Highway) would pass it and a curved railway would fail it. Replaced by the distance to road.
- **EDB school locations**: not used; School comes only from place labels.